<a href="https://colab.research.google.com/github/zienxu/CS3268/blob/main/notebooks/00_data_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# %% [markdown]
# # 00 — Data setup (W0)
# Run top to bottom once in Colab. Output: data/base.parquet and data/variant2.parquet in the shared Drive.
# Paste each "# %%" block into its own Colab cell.

In [ ]:
# %% 1. Mount Drive and make folders
from google.colab import drive
drive.mount('/content/drive')

import os
ROOT = "/content/drive/MyDrive/CS3268"
for sub in ["data", "preds", "models", "src"]:
    os.makedirs(f"{ROOT}/{sub}", exist_ok=True)

Mounted at /content/drive


In [ ]:
# %% 2. Download BAF from Kaggle
# If this asks for credentials: Kaggle > Settings > Create New Token, then upload kaggle.json.
# Fallback: download Base.csv and "Variant II.csv" by hand from
# https://www.kaggle.com/datasets/sgpjesus/bank-account-fraud-dataset-neurips-2022 and upload to ROOT/data.
!pip -q install kagglehub
import kagglehub
raw = kagglehub.dataset_download("sgpjesus/bank-account-fraud-dataset-neurips-2022")
print(os.listdir(raw))   # check the exact file names before the next cell


Using Colab cache for faster access to the 'bank-account-fraud-dataset-neurips-2022' dataset.
['Base.csv', 'Variant IV.csv', 'Variant V.csv', 'Variant I.csv', 'Variant III.csv', 'Variant II.csv']


In [ ]:
# %% 3. Load, add id and age_group, convert categoricals, save as parquet
import pandas as pd

CATEGORICALS = ["payment_type", "employment_status", "housing_status", "source", "device_os"]

def prepare(csv_path):
    df = pd.read_csv(csv_path)
    df.insert(0, "id", range(len(df)))                       # stable row id for every predictions file
    df["age_group"] = (df["customer_age"] >= 50).astype(int)  # 1 = 50-plus
    for c in CATEGORICALS:
        df[c] = df[c].astype("category")
    return df

base = prepare(f"{raw}/Base.csv")
v2 = prepare(f"{raw}/Variant II.csv")
base.to_parquet(f"{ROOT}/data/base.parquet", index=False)
v2.to_parquet(f"{ROOT}/data/variant2.parquet", index=False)

In [ ]:
# %% 4. Sanity checks — paste these numbers into the README
print(base.shape)                                   # expect about (1,000,000, 34)
print(sorted(base["customer_age"].unique()))        # expect decade bins: 10, 20, ..., 90
print(base[["id", "fraud_bool", "month", "age_group"]].dtypes)

split = base["month"].map(lambda m: "train" if m <= 4 else ("val" if m == 5 else "test"))
print(pd.crosstab(split, base["fraud_bool"], normalize="index").round(4))   # fraud ~1.1% each split
print(base.groupby("age_group")["fraud_bool"].mean().round(4))              # proposal cites 0.9% vs 1.8%
print(v2.groupby("age_group")["fraud_bool"].mean().round(4))                # proposal cites 0.4% vs 1.9%
print(split.value_counts())                                                  # rows per split

(1000000, 34)
[np.int64(10), np.int64(20), np.int64(30), np.int64(40), np.int64(50), np.int64(60), np.int64(70), np.int64(80), np.int64(90)]
id            int64
fraud_bool    int64
month         int64
age_group     int64
dtype: object
fraud_bool       0       1
month                     
test        0.9860  0.0140
train       0.9900  0.0100
val         0.9882  0.0118
age_group
0    0.0083
1    0.0234
Name: fraud_bool, dtype: float64
age_group
0    0.0036
1    0.0182
Name: fraud_bool, dtype: float64
month
train    675666
test     205011
val      119323
Name: count, dtype: int64


In [ ]:
# %% 5. Where do the -1 "missing" codes appear? (XGBoost can take them as-is; just record it)
neg = (base.select_dtypes("number") < 0).mean()
print(neg[neg > 0].round(3))

prev_address_months_count       0.713
current_address_months_count    0.004
intended_balcon_amount          0.743
velocity_6h                     0.000
credit_risk_score               0.014
bank_months_count               0.254
session_length_in_minutes       0.002
device_distinct_emails_8w       0.000
dtype: float64


In [ ]:
cols = ["prev_address_months_count","current_address_months_count","intended_balcon_amount",
        "velocity_6h","credit_risk_score","bank_months_count","session_length_in_minutes",
        "device_distinct_emails_8w"]
for c in cols:
    s = base[c]
    print(f"{c:32s} min={s.min():9.2f}  share==-1: {(s==-1).mean():.3f}  share<0: {(s<0).mean():.3f}")

prev_address_months_count        min=    -1.00  share==-1: 0.713  share<0: 0.713
current_address_months_count     min=    -1.00  share==-1: 0.004  share<0: 0.004
intended_balcon_amount           min=   -15.53  share==-1: 0.000  share<0: 0.743
velocity_6h                      min=  -170.60  share==-1: 0.000  share<0: 0.000
credit_risk_score                min=  -170.00  share==-1: 0.000  share<0: 0.014
bank_months_count                min=    -1.00  share==-1: 0.254  share<0: 0.254
session_length_in_minutes        min=    -1.00  share==-1: 0.002  share<0: 0.002
device_distinct_emails_8w        min=    -1.00  share==-1: 0.000  share<0: 0.000
